<a href="https://colab.research.google.com/github/nurallyssaroslan/KD344403-S2-25-26-G8-PROJECT/blob/main/Milestone4_ModelOptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, KFold #jacq
from catboost import CatBoostRegressor
from tqdm.notebook import tqdm
from joblib import parallel_backend
import contextlib
import joblib

# Progress bar helper for joblib/sklearn
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback

    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()


cb = CatBoostRegressor(
    verbose=0,
    random_state=42,
    allow_writing_files=False
)

param_dist = {
    "depth": [4, 5, 6, 7, 8, 9, 10, 11, 12],
    "learning_rate": [0.003, 0.005, 0.01, 0.02, 0.03, 0.05, 0.07, 0.1],
    "iterations": [200, 300, 500, 700, 1000, 1500],
    "l2_leaf_reg": [1, 3, 5, 7, 9, 12, 15],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "rsm": [0.6, 0.7, 0.8, 0.9, 1.0]
}

kf = KFold(n_splits=7, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=cb,
    param_distributions=param_dist,
    n_iter=100,
    cv=kf,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
    verbose=0
)

# Total fits = number of parameter combinations × number of folds
total_fits = 100 * 7

with tqdm_joblib(tqdm(desc="RandomizedSearchCV Progress", total=total_fits)):
    random_search.fit(X_train, y_train)

best_cb_random = random_search.best_estimator_

print("Best Random Search Params:")
print(random_search.best_params_)

print("Best Random Search CV R²:")
print(random_search.best_score_)

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold #jacq
from catboost import CatBoostRegressor
from tqdm.notebook import tqdm
import contextlib
import joblib
import numpy as np

@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback

    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()


cb_grid = CatBoostRegressor(
    verbose=0,
    random_state=42,
    allow_writing_files=False
)

focused_param_grid = {
    "depth": [10, 11, 12],
    "learning_rate": [0.03, 0.05, 0.07],
    "iterations": [600, 700, 800],
    "l2_leaf_reg": [2, 3, 5],
    "subsample": [0.6],
    "rsm": [0.8]
}

kf = KFold(n_splits=7, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=cb_grid,
    param_grid=focused_param_grid,
    cv=kf,
    scoring="r2",
    n_jobs=2,
    verbose=0
)

# Automatic total fits calculation
total_candidates = np.prod([len(values) for values in focused_param_grid.values()])
total_fits = total_candidates * kf.get_n_splits()

print(f"Total candidates: {total_candidates}")
print(f"Total fits: {total_fits}")

with tqdm_joblib(tqdm(desc="GridSearchCV Progress", total=total_fits)):
    grid_search.fit(X_train, y_train)

best_cb_grid = grid_search.best_estimator_

print("Best Grid Search Params:")
print(grid_search.best_params_)

print("Best Grid Search CV R²:")
print(grid_search.best_score_)

In [ ]:
from sklearn.model_selection import KFold #jacq
from catboost import CatBoostRegressor
import numpy as np
from tqdm.notebook import tqdm

# 7-fold CV for final tuned CatBoost model
kf = KFold(n_splits=7, shuffle=True, random_state=42)

tuned_scores = []

for train_idx, val_idx in tqdm(kf.split(X_train), total=7, desc="Tuned CatBoost 7-Fold CV"):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    tuned_model = CatBoostRegressor(
        depth=10,
        iterations=600,
        l2_leaf_reg=5,
        learning_rate=0.07,
        rsm=0.8,
        subsample=0.6,
        verbose=0,
        random_state=42,
        allow_writing_files=False
    )

    tuned_model.fit(X_tr, y_tr)
    score = tuned_model.score(X_val, y_val)  # R² score
    tuned_scores.append(score)

tuned_cv_r2 = np.mean(tuned_scores)
tuned_cv_std = np.std(tuned_scores)

print("Tuned CatBoost CV R² Mean:", tuned_cv_r2)
print("Tuned CatBoost CV R² Std:", tuned_cv_std)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

y_pred_tuned = best_cb_grid.predict(X_test)

tuned_mse = mean_squared_error(y_test, y_pred_tuned)
tuned_rmse = np.sqrt(tuned_mse)
tuned_r2 = r2_score(y_test, y_pred_tuned)

print("Tuned CatBoost Test R²:", tuned_r2)
print("Tuned CatBoost MSE:", tuned_mse)
print("Tuned CatBoost RMSE:", tuned_rmse)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Hyperparameters were taken from the best GridSearchCV result.
# The tuning process was not rerun here to save computational time.
# Fit final tuned model on full training set
best_cb_grid = CatBoostRegressor(
    depth=10,
    iterations=600,
    l2_leaf_reg=5,
    learning_rate=0.07,
    rsm=0.8,
    subsample=0.6,
    verbose=0,
    random_state=42,
    allow_writing_files=False
)

best_cb_grid.fit(X_train, y_train)

# Predict using tuned CatBoost
y_pred_tuned = best_cb_grid.predict(X_test)

# Tuned model test metrics
tuned_mse = mean_squared_error(y_test, y_pred_tuned)
tuned_rmse = np.sqrt(tuned_mse)
tuned_r2 = r2_score(y_test, y_pred_tuned)

# Get original CatBoost result before tuning
catboost_before = df_results[df_results["Model"] == "CatBoost"].iloc[0]

# Calculate tuned gap
tuned_gap = tuned_cv_r2 - tuned_r2

# Status rule
if tuned_cv_r2 < 0.2 and tuned_r2 < 0.2:
    tuned_status = "Underfitting"
elif tuned_gap > 0.05:
    tuned_status = "Overfitting"
else:
    tuned_status = "Good Fit"

comparison_tuning = pd.DataFrame([
    {
        "Model": "CatBoost Before Tuning",
        "CV_R2": catboost_before["CV_R2"],
        "CV_R2_STD": catboost_before["CV_R2_STD"],
        "Test_R2": catboost_before["Test_R2"],
        "MSE": catboost_before["MSE"],
        "RMSE": catboost_before["RMSE"],
        "Gap(CV-Test)": catboost_before["Gap(CV-Test)"],
        "Status": catboost_before["Status"]
    },
    {
        "Model": "CatBoost After Tuning",
        "CV_R2": round(tuned_cv_r2, 4),
        "CV_R2_STD": round(tuned_cv_std, 4),
        "Test_R2": round(tuned_r2, 4),
        "MSE": round(tuned_mse, 4),
        "RMSE": round(tuned_rmse, 4),
        "Gap(CV-Test)": round(tuned_gap, 4),
        "Status": tuned_status
    }
])

comparison_tuning